# Mjere centralnosti i klasteriranja

**Poglavlje:** [3. Mjere mrežne strukture i interpretacija rezultata](../content/03_mjere_mrezne_strukture.md)

U ovoj bilježnici **korak po korak** izračunavamo glavne mjere mrežne strukture na malom primjeru: **stupanj (degree)**, **betweenness**, **closeness**, **eigenvector** centralnost te **koeficijent klasteriranja** i **gustoću**. Za svaku mjeru prvo objasnimo što mjerimo, zatim je izračunamo i interpretiramo rezultate.

- **Definicije** svih mjera (stupanj, betweenness, closeness, eigenvector, klasteriranje, gustoća): [sadržaj pogl. 3 — Definicije](../content/03_mjere_mrezne_strukture.md#definicije).
- **Konceptualni primjeri** (filmovi, serije, umjetnici): [sadržaj pogl. 3 — Primjeri i interpretacija](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).

---

## Korak 1: Uvoz biblioteka i izgradnja primjer grafa

Koristit ćemo **NetworkX** za graf i **pandas** za pregled rezultata. Graf predstavlja malu društvenu mrežu: pet osoba (Ana, Bruno, Carla, David, Eva) i tko s kime je u kontaktu. **Ana, Bruno i Carla** međusobno se svi znaju (trokut). **David** poznaje Brunu i Carlu, a **Eva** poznaje samo Davida — dakle David je "most" između grupe (Ana,Bruno,Carla) i Eve. Takav graf jasno ilustrira razlike između mjera centralnosti.

*(U [sadržaju pogl. 3](../content/03_mjere_mrezne_strukture.md) ista ideja — "most između grupa" — ilustrirana je primjerima iz filmova/serija: npr. Nick Fury u MCU, Varys u Game of Thrones, Stringer Bell u The Wire.)*

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# Graf: čvorovi = osobe, bridovi = "poznaju se" / kontakt
# 1=Ana, 2=Bruno, 3=Carla, 4=David, 5=Eva
# Trokut: Ana–Bruno–Carla. David povezan s Bruno i Carlom. Eva samo s Davidom.
G = nx.Graph()
G.add_edges_from([
    (1, 2), (1, 3), (2, 3),   # trokut Ana–Bruno–Carla
    (2, 4), (3, 4),           # David s Bruno i Carlom
    (4, 5)                    # Eva samo s Davidom
])

# Imena za čitljivost (opcionalno za ispis)
names = {1: "Ana", 2: "Bruno", 3: "Carla", 4: "David", 5: "Eva"}
nx.set_node_attributes(G, names, "name")

# Zajednički raspored čvorova za sve kasnije crteže (reproducibilnost)
pos = nx.spring_layout(G, seed=42)

print("Broj čvorova:", G.number_of_nodes())
print("Broj veza:", G.number_of_edges())
print("Čvorovi i susjedi:")
for n in G.nodes():
    print(f"  {names.get(n, n)}: {[names.get(x, x) for x in G.neighbors(n)]}")

# Crtanje grafa: struktura "trokut + most" (teorija: čvorovi i veze)
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=600, node_color="steelblue", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Struktura mreže: trokut (Ana–Bruno–Carla) + most (David) + Eva")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 2: Stupanj centralnosti (degree)

**Što mjerimo:** Broj direktnih veza svakog čvora. Normalizirana verzija (`degree_centrality`) dijeli s maksimalno mogućim brojem veza (n−1), pa su vrijednosti između 0 i 1.  
**Pitanje na koje odgovara:** Tko ima najviše direktnih kontakata? Tko je "najpopularniji" u mreži?

*(Definicija: [pogl. 3 — Stupanj centralnosti](../content/03_mjere_mrezne_strukture.md#definicije). Konceptualni primjeri: Spider-Man/Harry Potter — "najviše veza" u [Primjerima iz filmova i serija](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Izračun: degree_centrality za svaki čvor
deg_cent = nx.degree_centrality(G)

print("Stupanj centralnosti (degree_centrality):")
for node, val in sorted(deg_cent.items(), key=lambda x: -x[1]):
    print(f"  {names[node]:6s}: {val:.4f}  (broj susjeda: {G.degree(node)})")

# Tko ima najviši stupanj?
max_node = max(deg_cent, key=deg_cent.get)
print(f"\nNajviši stupanj: {names[max_node]} — ima najviše direktnih veza.")

# Graf: veličina čvora = degree (teorija: "najviše veza" = veći čvor)
node_sizes = [deg_cent[n] * 2500 + 400 for n in G.nodes()]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=node_sizes, node_color="steelblue", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Degree centralnost: veličina ∝ broj direktnih veza")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 3: Posrednička centralnost (betweenness)

**Što mjerimo:** Za svaki par drugih čvorova računamo najkraći put; betweenness čvora je koliki *udio* tih najkraćih puteva *prolazi kroz* taj čvor.  
**Pitanje na koje odgovara:** Tko je "most" između grupa? Tko kontrolira protok informacija između drugih?

*(Definicija: [pogl. 3 — Posrednička centralnost](../content/03_mjere_mrezne_strukture.md#definicije). U sadržaju: Nick Fury, Varys/Littlefinger, Stringer Bell — [Primjeri iz filmova i serija](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Izračun: betweenness_centrality
bet_cent = nx.betweenness_centrality(G)

print("Posrednička centralnost (betweenness):")
for node, val in sorted(bet_cent.items(), key=lambda x: -x[1]):
    print(f"  {names[node]:6s}: {val:.4f}")

# Zašto je David (4) visok? Ana–Eva, Bruno–Eva, Carla–Eva idu SVE preko Davida.
print("\nDavid ima visok betweenness jer svi putevi iz trokuta (Ana,Bruno,Carla) do Eve prolaze kroz njega — on je 'most'.")

# Graf: veličina čvora = betweenness (teorija: "most" = najveći čvor)
node_sizes = [bet_cent[n] * 4000 + 400 for n in G.nodes()]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=node_sizes, node_color="coral", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Betweenness: veličina ∝ udio najkraćih puteva kroz čvor (David = most)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 4: Blizinska centralnost (closeness)

**Što mjerimo:** Koliko je čvor "blizu" svih ostalih — obično recipročna vrijednost prosječne duljine najkraćeg puta do svih drugih čvorova. Što je viša, to čvor brže "dosegne" cijelu mrežu (ili je brzo dosegnut).  
**Pitanje na koje odgovara:** Tko može najbrže širiti informaciju na cijelu mrežu? Tko je u "sredini" u smislu udaljenosti?

*(Definicija: [pogl. 3 — Blizinska centralnost](../content/03_mjere_mrezne_strukture.md#definicije). Primjeri u sadržaju: Dumbledore, Tony Stark — [Primjeri iz filmova i serija](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Izračun: closeness_centrality
clo_cent = nx.closeness_centrality(G)

print("Blizinska centralnost (closeness):")
for node, val in sorted(clo_cent.items(), key=lambda x: -x[1]):
    print(f"  {names[node]:6s}: {val:.4f}")

# Opcionalno: prosječna duljina puta (da vidimo što closeness "vidi")
print("\nProsječna duljina najkraćeg puta do ostalih (manje = 'bliže' cijeloj mreži):")
for node in G.nodes():
    lengths = [nx.shortest_path_length(G, node, other) for other in G.nodes() if other != node]
    avg = sum(lengths) / len(lengths) if lengths else 0
    print(f"  {names[node]:6s}: {avg:.2f}")

# Graf: veličina čvora = closeness (teorija: "blizu svima" = veći čvor)
node_sizes = [clo_cent[n] * 3500 + 400 for n in G.nodes()]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=node_sizes, node_color="seagreen", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Closeness: veličina ∝ blizina ostalima (brzo dosegnut / dosegnuti)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 5: Središnja centralnost (eigenvector)

**Što mjerimo:** Važnost čvora ovisi o važnosti njegovih susjeda — "biti povezan s važnima" podiže tvoju eigenvector centralnost. (Slično PageRanku.)  
**Pitanje na koje odgovara:** Tko je "važan" jer je povezan s drugima važnima, a ne nužno jer ima najviše veza?

*(Definicija: [pogl. 3 — Središnja centralnost (eigenvector)](../content/03_mjere_mrezne_strukture.md#definicije). Primjeri: MCU novi lik s Avengersima, Harry–Dumbledore/Snape; umjetnost — [Primjeri iz umjetnosti i književnosti](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Izračun: eigenvector_centrality (NetworkX koristi iteraciju; za povezane grafove konvergira)
eig_cent = nx.eigenvector_centrality(G)

print("Središnja centralnost (eigenvector):")
for node, val in sorted(eig_cent.items(), key=lambda x: -x[1]):
    print(f"  {names[node]:6s}: {val:.4f}")

print("\nČvorovi u 'gušćem' dijelu mreže (trokut) obično imaju višu eigenvector centralnost.")

# Graf: veličina čvora = eigenvector (teorija: "važan kroz važne susjede" = veći čvor)
node_sizes = [eig_cent[n] * 3500 + 400 for n in G.nodes()]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=node_sizes, node_color="darkviolet", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Eigenvector: veličina ∝ važnost susjeda (trokut = gušći = viša vrijednost)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 6: Koeficijent klasteriranja (lokalni i globalni)

**Što mjerimo:** Za svaki čvor — koliki dio *mogućih* veza *između njegovih susjeda* je zapravo ostvaren? Ako su svi susjedi međusobno povezani, koeficijent je 1; ako nema veza između susjeda, 0. **Globalni** koeficijent je prosjek po čvorovima.  
**Pitanje na koje odgovara:** Ima li mreža "grupe" u kojima su ljudi gusto povezani? (npr. "prijatelji mojih prijatelja su i moji prijatelji")

*(Definicija: [pogl. 3 — Lokalni/globalni koeficijent klasteriranja](../content/03_mjere_mrezne_strukture.md#definicije). Primjeri: Hogwarts kuće, Avengers vs Guardians, Bloomsbury, impresionisti — [Primjeri u sadržaju](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Lokalni koeficijent klasteriranja po čvoru
clust = nx.clustering(G)

print("Lokalni koeficijent klasteriranja:")
for node, val in sorted(clust.items(), key=lambda x: -x[1]):
    print(f"  {names[node]:6s}: {val:.4f}  (susjedi međusobno: {val*100:.0f}% mogućih veza)")

# Globalni koeficijent klasteriranja (cijela mreža)
global_clust = nx.average_clustering(G)
print(f"\nGlobalni (prosječni) koeficijent klasteriranja mreže: {global_clust:.4f}")

# Zašto Ana, Bruno, Carla imaju 1? Trokut = svi susjedi međusobno povezani. Eva ima 0 jer ima samo jednog susjeda (nema par susjeda).

# Graf: veličina čvora = koeficijent klasteriranja (teorija: trokut = 1 = najveći čvorovi)
node_sizes = [clust[n] * 2500 + 400 for n in G.nodes()]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=node_sizes, node_color="darkorange", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title("Klasteriranje: veličina ∝ susjedi međusobno povezani (trokut Ana–Bruno–Carla = 1)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 7: Gustoća mreže

**Što mjerimo:** Omjer broja *stvarnih* veza i broja *mogućih* veza (kad bi svaki čvor bio povezan sa svakim). Vrijednost između 0 i 1; viša = gušća, kohezivnija mreža.

*(Definicija i interpretacija: [pogl. 3 — Gustoća](../content/03_mjere_mrezne_strukture.md#definicije); primjeri gustoće u [Primjerima iz umjetnosti](../content/03_mjere_mrezne_strukture.md#primjeri-i-interpretacija).)*

In [ ]:
# Gustoća = stvarne veze / moguće veze. Za n čvorova: najviše n*(n-1)/2 bridova (neusmjereni graf).
density = nx.density(G)
n = G.number_of_nodes()
max_edges = n * (n - 1) / 2
actual_edges = G.number_of_edges()

print(f"Broj čvorova: {n}, broj veza: {actual_edges}, maksimalno mogućih: {max_edges:.0f}")
print(f"Gustoća mreže: {density:.4f}  (= {actual_edges}/{max_edges:.0f})")
print("  (0 = nema veza, 1 = svi povezani sa svima)")

# Graf: isti raspored, naslov objašnjava gustoću (teorija: mjera kohezije cijele mreže)
plt.figure(figsize=(5, 4))
nx.draw(G, pos, node_size=600, node_color="teal", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=10)
plt.title(f"Gustoća mreže = {density:.2f} (kohezija: {actual_edges} od {max_edges:.0f} mogućih veza)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Korak 8: Pregled svih mjera u jednoj tablici

Sve mjere možemo staviti u jednu tablicu (DataFrame) i usporediti čvorove. Korisno je i vizualizirati graf — npr. veličina čvora prema odabranoj centralnosti. U [sadržaju pogl. 3](../content/03_mjere_mrezne_strukture.md) na kraju je tablica "Povezivanje sadržaj ↔ kod" koja pokazuje gdje što tražiti (definicije, kod, primjeri).

In [ ]:
# Sažeta tablica svih mjera (koristeći već izračunate vrijednosti)
df = pd.DataFrame({
    "čvor": [names[n] for n in G.nodes()],
    "degree": [deg_cent[n] for n in G.nodes()],
    "betweenness": [bet_cent[n] for n in G.nodes()],
    "closeness": [clo_cent[n] for n in G.nodes()],
    "eigenvector": [eig_cent[n] for n in G.nodes()],
    "clustering": [clust[n] for n in G.nodes()],
})
df = df.set_index("čvor")
print("Pregled mjera po čvoru:\n")
print(df.round(4))

---

## Korak 9: Pregled — graf za svaku mjeru (teorija u praksi)

U jednom prikazu crtamo graf **pet puta**: veličina čvora odražava redom **degree**, **betweenness**, **closeness**, **eigenvector** i **clustering**. Tako se vidi kako ista struktura (trokut + most) daje različite "središnjosti" ovisno o mjeri.

In [ ]:
# Pregled: isti graf, veličina čvora = različita mjera (teorija ilustrirana za svaku mjeru)
measures = [
    (deg_cent, "Degree\n(broj veza)", "steelblue"),
    (bet_cent, "Betweenness\n(most)", "coral"),
    (clo_cent, "Closeness\n(blizina svima)", "seagreen"),
    (eig_cent, "Eigenvector\n(važnost susjeda)", "darkviolet"),
    (clust, "Clustering\n(trokut = 1)", "darkorange"),
]
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()
for i, (vals, title, color) in enumerate(measures):
    ax = axes[i]
    node_sizes = [vals[n] * 2500 + 400 for n in G.nodes()]
    nx.draw(G, pos, ax=ax, node_size=node_sizes, node_color=color, with_labels=False)
    nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=8, ax=ax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
# Zadnji panel: osnovni graf + gustoća
axes[5].set_title(f"Gustoća = {density:.2f}\n(kohezija mreže)", fontsize=10)
nx.draw(G, pos, ax=axes[5], node_size=500, node_color="teal", with_labels=False)
nx.draw_networkx_labels(G, pos, {n: names[n] for n in G.nodes()}, font_size=8, ax=axes[5])
axes[5].axis("off")
plt.suptitle("Isti graf — veličina čvora ilustrira teoriju za svaku mjeru", fontsize=11)
plt.tight_layout()
plt.show()

---

## Zaključak

- **Degree** — tko ima najviše direktnih veza (u primjeru: Bruno, Carla, David).
- **Betweenness** — tko leži na putovima između drugih; David je "most" između trokuta (Ana, Bruno, Carla) i Eve.
- **Closeness** — tko je u prosjeku "blizu" svima (često čvorovi u sredini strukture).
- **Eigenvector** — važnost kroz važne susjede; čvorovi u gušćem dijelu mreže (trokut) imaju višu vrijednost.
- **Klasteriranje** — Ana, Bruno, Carla imaju 1 (trokut); Eva ima 0 (samo jedan susjed).

U praksi odabir mjere ovisi o istraživačkom pitanju. Za "tko je most?" koristimo betweenness; za "tko ima najviše kontakata?" degree. Uvijek interpretirajte brojke u kontekstu podataka i teorije.

**Sadržaj ↔ kod:** Za definicije i konceptualne primjere (filmovi, serije, umjetnici, književnost, društvene mreže) vidi [03_mjere_mrezne_strukture.md](../content/03_mjere_mrezne_strukture.md). Tablica "Povezivanje sadržaj ↔ kod" na kraju tog dokumenta usmjeruje na odgovarajuće korake u ovoj bilježnici.